In [ ]:
# ------------------------
# ライブラリのインストール
# Google Colab / Python 3.12対応
# ------------------------
import sys
import subprocess

# yt-dlpの音声変換にはPython版ffmpegではなく、
# OSのffmpeg実行ファイルが必要
subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    ["apt-get", "install", "-y", "-qq", "ffmpeg"],
    check=True
)

# Python 3.12対応バージョンを使用
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-U",
        "pip",
        "setuptools",
        "wheel",
        "yt-dlp[default]",
        "librosa>=0.10.2,<0.12",
        "numpy>=1.26,<3",
        "matplotlib",
        "scipy",
        "soundfile",
        # 文字起こし・自然言語処理・可視化
        "faster-whisper>=1.1.0",
        "wordcloud>=1.9.3",
        "janome>=0.5.0",
        "sentence-transformers>=3.0.0",
        "scikit-learn>=1.4",
        "pandas>=2.2",
    ],
    check=True
)

# 日本語ワードクラウド用フォント
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"],
    check=True,
)

print("Installation completed.")

In [ ]:
# Program Name: youtube_audio_analysis_with_tempo.py
# Python 3.12 / Google Colab対応版

# ------------------------
# パラメータ定義
# ------------------------
from pathlib import Path
import subprocess
import sys

url = "https://www.youtube.com/watch?v=kt5-Al0CMuk"

wav_path = Path("audio.wav")
harmonic_path = Path("harmonic.wav")
percussive_path = Path("percussive.wav")
output_image_path = Path("analysis_with_tempo.png")

sr = 22050
frame_length = 2048
hop_length = 512

# RMSピーク検出設定
peak_threshold = 0.05
minimum_peak_interval_sec = 0.20

# ------------------------
# YouTube音声ダウンロード
# yt-dlpから直接WAVへ変換
# ------------------------
download_command = [
    sys.executable,
    "-m",
    "yt_dlp",
    "-x",
    "--audio-format",
    "wav",
    "--audio-quality",
    "0",
    "--force-overwrites",
    "-o",
    str(wav_path),
    url,
]

subprocess.run(download_command, check=True)

if not wav_path.exists():
    raise FileNotFoundError(
        f"WAVファイルが作成されませんでした: {wav_path.resolve()}"
    )

print(f"Audio downloaded: {wav_path.resolve()}")

# ------------------------
# ライブラリ読み込み
# ------------------------
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

from scipy.signal import find_peaks

# ------------------------
# 音声読み込み
# ------------------------
y, sr_actual = librosa.load(
    wav_path,
    sr=sr,
    mono=True,
)

if y.size == 0:
    raise ValueError("読み込んだ音声データが空です。")

duration = librosa.get_duration(y=y, sr=sr_actual)

# ------------------------
# STFT・スペクトログラム
# ------------------------
stft_complex = librosa.stft(
    y,
    n_fft=frame_length,
    hop_length=hop_length,
)

magnitude = np.abs(stft_complex)
power_spectrogram = magnitude**2

spectrogram_db = librosa.power_to_db(
    power_spectrogram,
    ref=np.max,
)

# ------------------------
# ピッチ抽出
# ------------------------
f0, voiced_flag, voiced_probability = librosa.pyin(
    y,
    fmin=librosa.note_to_hz("C2"),
    fmax=librosa.note_to_hz("C7"),
    sr=sr_actual,
    frame_length=frame_length,
    hop_length=hop_length,
)

# ------------------------
# スペクトルエントロピー
# 各フレームを確率分布として正規化
# ------------------------
power_sum = np.sum(
    power_spectrogram,
    axis=0,
    keepdims=True,
)

probability_spectrum = power_spectrogram / np.maximum(
    power_sum,
    np.finfo(float).eps,
)

entropy = -np.sum(
    probability_spectrum
    * np.log2(probability_spectrum + np.finfo(float).eps),
    axis=0,
)

# 周波数ビン数によって0〜1へ正規化
normalized_entropy = entropy / np.log2(
    power_spectrogram.shape[0]
)

# ------------------------
# 音源分離
# ------------------------
harmonic, percussive = librosa.effects.hpss(y)

sf.write(harmonic_path, harmonic, sr_actual)
sf.write(percussive_path, percussive, sr_actual)

# ------------------------
# テンポ推定
# ------------------------
tempo_result, beats = librosa.beat.beat_track(
    y=percussive,
    sr=sr_actual,
    hop_length=hop_length,
)

# librosaのバージョンによってfloatまたは配列になるため安全に変換
tempo = float(np.asarray(tempo_result).reshape(-1)[0])

beat_times = librosa.frames_to_time(
    beats,
    sr=sr_actual,
    hop_length=hop_length,
)

print(f"Estimated Tempo: {tempo:.2f} BPM")
print(f"Beat Count: {len(beats)}")

# ------------------------
# 音量ピーク検出
#
# 元コードの np.where(abs(y) > threshold) は、
# ピーク数ではなく閾値を超えた全サンプル数を数えてしまうため、
# RMSエネルギーとfind_peaksを使用
# ------------------------
rms = librosa.feature.rms(
    y=y,
    frame_length=frame_length,
    hop_length=hop_length,
)[0]

rms_times = librosa.frames_to_time(
    np.arange(len(rms)),
    sr=sr_actual,
    hop_length=hop_length,
)

minimum_peak_distance_frames = max(
    1,
    int(
        minimum_peak_interval_sec
        * sr_actual
        / hop_length
    ),
)

peak_indices, peak_properties = find_peaks(
    rms,
    height=peak_threshold,
    distance=minimum_peak_distance_frames,
)

peak_times = rms_times[peak_indices]
peak_heights = rms[peak_indices]

# 描画用ピーク数制限
max_peaks_to_plot = 1000

if len(peak_indices) > max_peaks_to_plot:
    selection = np.linspace(
        0,
        len(peak_indices) - 1,
        max_peaks_to_plot,
    ).astype(int)

    peak_indices_vis = peak_indices[selection]
    peak_times_vis = peak_times[selection]
    peak_heights_vis = peak_heights[selection]
else:
    peak_indices_vis = peak_indices
    peak_times_vis = peak_times
    peak_heights_vis = peak_heights

# ------------------------
# FFT
# 長時間音源では巨大な値になりやすいため正規化
# ------------------------
window = np.hanning(len(y))
fft_values = np.abs(np.fft.rfft(y * window))

fft_values = fft_values / max(
    1,
    len(y),
)

fft_frequencies = np.fft.rfftfreq(
    len(y),
    d=1 / sr_actual,
)

# 解析しやすいよう20 kHzまで表示
fft_max_frequency = min(
    20000,
    sr_actual / 2,
)

fft_mask = fft_frequencies <= fft_max_frequency

# ------------------------
# 可視化
# ------------------------
fig, axs = plt.subplots(
    3,
    1,
    figsize=(12, 10),
    dpi=120,
)

# 1. Spectrogram
img = librosa.display.specshow(
    spectrogram_db,
    sr=sr_actual,
    hop_length=hop_length,
    x_axis="time",
    y_axis="log",
    ax=axs[0],
)

axs[0].set_title("Spectrogram")
fig.colorbar(
    img,
    ax=axs[0],
    format="%+2.0f dB",
)

# 2. Waveform + volume peaks
librosa.display.waveshow(
    y,
    sr=sr_actual,
    ax=axs[1],
    alpha=0.7,
    linewidth=0.5,
    label="Waveform",
)

if len(peak_times_vis) > 0:
    axs[1].vlines(
        peak_times_vis,
        ymin=np.min(y),
        ymax=np.max(y),
        alpha=0.35,
        linewidth=0.6,
        label="RMS peaks",
    )

axs[1].set_title("Waveform and RMS Volume Peaks")
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Amplitude")
axs[1].legend(loc="upper right")

# 3. FFT
axs[2].plot(
    fft_frequencies[fft_mask],
    fft_values[fft_mask],
    linewidth=0.6,
)

axs[2].set_title("FFT Spectrum")
axs[2].set_xlabel("Frequency (Hz)")
axs[2].set_ylabel("Normalized amplitude")
axs[2].set_xlim(0, fft_max_frequency)

plt.tight_layout()
plt.savefig(
    output_image_path,
    bbox_inches="tight",
)
plt.show()

# ------------------------
# 結果出力
# ------------------------
valid_f0 = f0[~np.isnan(f0)]

print("\nAnalysis summary")
print(f"Duration: {duration:.2f} sec")
print(f"Sample rate: {sr_actual} Hz")
print(f"Voiced frames: {len(valid_f0)}")
print(f"Beat count: {len(beats)}")
print(f"Volume peaks: {len(peak_times)}")
print(f"Peak threshold: {peak_threshold:.4f}")
print(f"Estimated tempo: {tempo:.2f} BPM")

if valid_f0.size > 0:
    print(f"Pitch mean: {np.mean(valid_f0):.2f} Hz")
    print(f"Pitch median: {np.median(valid_f0):.2f} Hz")
    print(f"Pitch standard deviation: {np.std(valid_f0):.2f} Hz")
else:
    print("Pitch: No voiced frames detected")

print(
    "Normalized spectral entropy: "
    f"mean={np.mean(normalized_entropy):.4f}, "
    f"std={np.std(normalized_entropy):.4f}"
)

print(f"Harmonic audio: {harmonic_path.resolve()}")
print(f"Percussive audio: {percussive_path.resolve()}")
print(f"Analysis image: {output_image_path.resolve()}")

# 音声文字起こし・ワードクラウド・意味クラスタリング
上のセルで取得・解析した `audio.wav` を再利用します。Whisperで文字起こしし、頻出語をワードクラウド化し、発話区間を意味ベクトルでクラスタ分類します。


In [ ]:
# ------------------------
# 文字起こし・ワードクラウド・意味クラスタリング
# ------------------------
from pathlib import Path
from collections import Counter
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from faster_whisper import WhisperModel
from janome.tokenizer import Tokenizer
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from wordcloud import WordCloud

# ------------------------
# 設定
# ------------------------
# tiny / base / small / medium / large-v3
# Colabの実行速度と精度のバランスからsmallを既定値にしています。
whisper_model_size = "small"
transcription_language = "ja"  # 自動判定はNone
beam_size = 5

# クラスタ数。Noneならシルエット係数から自動選択します。
requested_cluster_count = None
max_cluster_count = 8

transcript_txt_path = Path("transcript.txt")
transcript_json_path = Path("transcript_segments.json")
word_frequency_csv_path = Path("word_frequencies.csv")
wordcloud_image_path = Path("wordcloud.png")
cluster_csv_path = Path("transcript_clusters.csv")
cluster_image_path = Path("transcript_clusters.png")

if not wav_path.exists():
    raise FileNotFoundError(f"音声ファイルがありません: {wav_path.resolve()}")

# ------------------------
# Whisperによる文字起こし
# ------------------------
# GPUが使える場合は自動的にGPUを利用し、なければCPUへフォールバックします。
try:
    import torch
    use_cuda = torch.cuda.is_available()
except Exception:
    use_cuda = False

device = "cuda" if use_cuda else "cpu"
compute_type = "float16" if use_cuda else "int8"

print(f"Whisper device: {device} / compute_type: {compute_type}")
model = WhisperModel(
    whisper_model_size,
    device=device,
    compute_type=compute_type,
)

segments_generator, transcription_info = model.transcribe(
    str(wav_path),
    language=transcription_language,
    beam_size=beam_size,
    vad_filter=True,
    word_timestamps=True,
)

segments = []
for segment in segments_generator:
    text = segment.text.strip()
    if not text:
        continue
    segments.append({
        "id": int(segment.id),
        "start": float(segment.start),
        "end": float(segment.end),
        "text": text,
        "words": [
            {
                "start": None if word.start is None else float(word.start),
                "end": None if word.end is None else float(word.end),
                "word": word.word.strip(),
                "probability": float(word.probability),
            }
            for word in (segment.words or [])
        ],
    })

if not segments:
    raise ValueError("文字起こし結果が空です。音声に明瞭な発話があるか確認してください。")

full_transcript = "\n".join(item["text"] for item in segments)
transcript_txt_path.write_text(full_transcript, encoding="utf-8")
transcript_json_path.write_text(
    json.dumps(
        {
            "language": transcription_info.language,
            "language_probability": float(transcription_info.language_probability),
            "duration": float(transcription_info.duration),
            "segments": segments,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(f"Detected language: {transcription_info.language}")
print(f"Language probability: {transcription_info.language_probability:.3f}")
print(f"Transcript segments: {len(segments)}")
print("\n--- Transcript preview ---")
print(full_transcript[:1000])

# ------------------------
# 日本語形態素解析・頻出語抽出
# ------------------------
tokenizer = Tokenizer()

# 内容語中心。必要に応じて追加・削除してください。
target_pos = {"名詞", "動詞", "形容詞", "副詞"}
stop_words = {
    "する", "ある", "いる", "なる", "れる", "られる", "こと", "もの",
    "これ", "それ", "あれ", "ここ", "そこ", "ため", "よう", "そう",
    "さん", "ん", "の", "です", "ます", "ない", "という", "思う",
}

def extract_terms(text):
    terms = []
    for token in tokenizer.tokenize(text):
        surface = token.surface.strip()
        base = token.base_form if token.base_form != "*" else surface
        main_pos = token.part_of_speech.split(",")[0]

        if main_pos not in target_pos:
            continue
        if len(surface) <= 1:
            continue
        if base in stop_words or surface in stop_words:
            continue
        if re.fullmatch(r"[\W_]+", surface):
            continue
        if re.fullmatch(r"\d+", surface):
            continue
        terms.append(base)
    return terms

terms = extract_terms(full_transcript)
word_counts = Counter(terms)

if not word_counts:
    raise ValueError("ワードクラウドに使える単語を抽出できませんでした。")

frequency_df = pd.DataFrame(
    word_counts.most_common(),
    columns=["word", "count"],
)
frequency_df.to_csv(word_frequency_csv_path, index=False, encoding="utf-8-sig")

display(frequency_df.head(30))

# ------------------------
# ワードクラウド
# ------------------------
font_candidates = [
    Path("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"),
    Path("/usr/share/fonts/opentype/noto/NotoSerifCJK-Regular.ttc"),
]
font_path = next((path for path in font_candidates if path.exists()), None)
if font_path is None:
    raise FileNotFoundError("日本語フォントが見つかりません。fonts-noto-cjkを再インストールしてください。")

wordcloud = WordCloud(
    width=1600,
    height=900,
    background_color="white",
    font_path=str(font_path),
    max_words=200,
    collocations=False,
).generate_from_frequencies(word_counts)

plt.figure(figsize=(14, 8), dpi=120)
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Transcript Word Cloud")
plt.tight_layout()
plt.savefig(wordcloud_image_path, bbox_inches="tight")
plt.show()

# ------------------------
# 発話区間の意味クラスタリング
# ------------------------
segment_texts = [item["text"] for item in segments]

# 多言語対応の文埋め込みモデル。日本語にも対応します。
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
embeddings = embedding_model.encode(
    segment_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)

sample_count = len(segment_texts)

# クラスタ数を決定
if sample_count < 3:
    cluster_count = 1
elif requested_cluster_count is not None:
    cluster_count = max(2, min(int(requested_cluster_count), sample_count - 1))
else:
    upper = min(max_cluster_count, sample_count - 1)
    candidates = range(2, upper + 1)
    scored_candidates = []

    for k in candidates:
        candidate_model = KMeans(n_clusters=k, random_state=42, n_init="auto")
        candidate_labels = candidate_model.fit_predict(embeddings)
        score = silhouette_score(embeddings, candidate_labels, metric="cosine")
        scored_candidates.append((score, k))

    cluster_count = max(scored_candidates)[1]

if cluster_count == 1:
    labels = np.zeros(sample_count, dtype=int)
else:
    cluster_model = KMeans(
        n_clusters=cluster_count,
        random_state=42,
        n_init="auto",
    )
    labels = cluster_model.fit_predict(embeddings)

# クラスタごとの代表語を抽出
cluster_keywords = {}
for cluster_id in sorted(set(labels)):
    cluster_text = " ".join(
        text for text, label in zip(segment_texts, labels)
        if label == cluster_id
    )
    cluster_terms = Counter(extract_terms(cluster_text))
    cluster_keywords[int(cluster_id)] = [
        word for word, _ in cluster_terms.most_common(5)
    ]

cluster_df = pd.DataFrame({
    "segment_id": [item["id"] for item in segments],
    "start_sec": [item["start"] for item in segments],
    "end_sec": [item["end"] for item in segments],
    "cluster": labels.astype(int),
    "cluster_keywords": [
        " / ".join(cluster_keywords[int(label)])
        for label in labels
    ],
    "text": segment_texts,
})
cluster_df.to_csv(cluster_csv_path, index=False, encoding="utf-8-sig")

display(cluster_df)

# ------------------------
# クラスタの2次元可視化
# ------------------------
if sample_count >= 2:
    pca = PCA(n_components=2, random_state=42)
    points_2d = pca.fit_transform(embeddings)

    plt.figure(figsize=(11, 8), dpi=120)
    scatter = plt.scatter(
        points_2d[:, 0],
        points_2d[:, 1],
        c=labels,
        s=70,
        alpha=0.8,
        cmap="tab10",
    )

    for index, (x, y_point) in enumerate(points_2d):
        plt.annotate(
            str(index),
            (x, y_point),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
        )

    plt.title("Transcript Semantic Clusters (PCA projection)")
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.colorbar(scatter, label="Cluster")
    plt.tight_layout()
    plt.savefig(cluster_image_path, bbox_inches="tight")
    plt.show()
else:
    print("発話区間が1件のため、クラスタ散布図は作成しません。")

# ------------------------
# 出力ファイル一覧
# ------------------------
print("\nText analysis summary")
print(f"Transcript: {transcript_txt_path.resolve()}")
print(f"Transcript JSON: {transcript_json_path.resolve()}")
print(f"Word frequencies: {word_frequency_csv_path.resolve()}")
print(f"Word cloud: {wordcloud_image_path.resolve()}")
print(f"Cluster table: {cluster_csv_path.resolve()}")
if sample_count >= 2:
    print(f"Cluster image: {cluster_image_path.resolve()}")
print(f"Selected cluster count: {cluster_count}")
print(f"Top words: {word_counts.most_common(20)}")
